# Oil Production Prediction - Complete Training Pipeline

This notebook reproduces our best-performing model (~1.95M RMSE) for the Energy A.I. Hackathon 2026.

**Pipeline Overview:**
1. Data Loading
2. Feature Engineering (Physics + Spatial kNN)
3. Model Training (Ensemble)
4. Evaluation
5. Generate Predictions

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy pandas scikit-learn xgboost lightgbm torch matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
from pathlib import Path

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV
import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
import torch.nn.functional as F

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("Setup complete!")

## 2. Data Loading

In [ ]:
# File paths (adjust as needed)
DATA_DIR = Path(".")

prod_logs = pd.read_csv(DATA_DIR / "Well_log_data_production_wells.csv")
pre_logs = pd.read_csv(DATA_DIR / "Well_log_data_preproduction_wells.csv")
prod_hist = pd.read_csv(DATA_DIR / "Production_history_production_wells.csv")
sand_map = np.load(DATA_DIR / "2d_sand_proportion.npy")

print(f"Production logs: {prod_logs.shape}")
print(f"Pre-production logs: {pre_logs.shape}")
print(f"Production history: {prod_hist.shape}")
print(f"Sand map: {sand_map.shape}")

## 3. Target Engineering

Calculate 3-year cumulative oil increment (Dec 2022 → Nov 2025)

In [ ]:
prod_hist = prod_hist.copy()
prod_hist["Date"] = pd.to_datetime(prod_hist["Date"])

START = pd.Timestamp("2022-12-01")
END = pd.Timestamp("2025-11-01")
oil_col = "Cumulative Oil Production, BBL"

def cum_at_or_before(df, t):
    d = df[df["Date"] <= t]
    return float(d.iloc[-1][oil_col]) if len(d) > 0 else 0.0

ys = []
for wid, g in prod_hist.sort_values("Date").groupby("Well_ID"):
    start_cum = cum_at_or_before(g, START)
    end_cum = cum_at_or_before(g, END)
    ys.append((wid, max(0.0, end_cum - start_cum)))

y_df = pd.DataFrame(ys, columns=["Well_ID", "y_3yr_cum_oil"])
print(f"Target shape: {y_df.shape}")
print(y_df["y_3yr_cum_oil"].describe())

## 4. Feature Engineering

### 4.1 Physics Features

In [ ]:
def add_physics_features(df):
    """Add high-signal rock-physics ratios."""
    out = df.copy()
    eps = 1e-9
    
    # Vp/Vs Ratio
    if {"Vp", "Vs"}.issubset(out.columns):
        out["Vp_Vs"] = out["Vp"] / (out["Vs"] + eps)
        
        # Poisson's ratio
        r = out["Vp_Vs"]
        r2 = r * r
        denom = (r2 - 1.0)
        out["poisson"] = np.where(np.abs(denom) > 1e-6, 0.5 * (r2 - 2.0) / denom, np.nan)
    
    # Elastic moduli
    if {"Vp", "Vs", "rho_b"}.issubset(out.columns):
        rho = out["rho_b"].astype(float)
        vp = out["Vp"].astype(float)
        vs = out["Vs"].astype(float)
        out["mu_rho"] = (vs**2) * rho
        out["lambda_rho"] = (vp**2 - 2*(vs**2)) * rho
    
    # Brittleness proxy
    if {"Vs", "poisson"}.issubset(out.columns):
        out["brittle_proxy"] = out["Vs"] / (out["poisson"].clip(lower=0.05, upper=0.45) + eps)
    
    return out

print("Physics features function defined.")

### 4.2 Well-Level Aggregation

In [ ]:
def aggregate_well_features(log_df):
    """Aggregate depth-level logs to well-level features."""
    log_df = add_physics_features(log_df)
    
    id_cols = {"Well_ID", "X", "Y", "Depth_m", "Z"}
    num_cols = [c for c in log_df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(log_df[c])]
    
    feats = []
    for wid, g in log_df.groupby("Well_ID"):
        row = {"Well_ID": wid}
        row["X"] = float(g["X"].iloc[0]) if "X" in g.columns else np.nan
        row["Y"] = float(g["Y"].iloc[0]) if "Y" in g.columns else np.nan
        
        for c in num_cols:
            s = g[c].astype(float)
            row[f"{c}_mean"] = float(s.mean())
            row[f"{c}_std"] = float(s.std())
            row[f"{c}_p10"] = float(np.nanpercentile(s, 10))
            row[f"{c}_p90"] = float(np.nanpercentile(s, 90))
        
        feats.append(row)
    
    return pd.DataFrame(feats)

# Aggregate
X_prod = aggregate_well_features(prod_logs)
X_pre = aggregate_well_features(pre_logs)

print(f"Production features: {X_prod.shape}")
print(f"Pre-production features: {X_pre.shape}")

### 4.3 Sand Map Features

In [ ]:
def add_sand_features(df, sand_map, x_min, x_max, y_min, y_max):
    """Add sand map value and neighborhood mean."""
    df = df.copy()
    ny, nx = sand_map.shape
    
    def to_idx(val, vmin, vmax, n):
        if vmax == vmin:
            return 0
        scaled = (val - vmin) / (vmax - vmin) * (n - 1)
        return int(np.clip(int(np.round(scaled)), 0, n-1))
    
    vals, means = [], []
    for _, row in df.iterrows():
        x, y = row.get("X", np.nan), row.get("Y", np.nan)
        if pd.isna(x) or pd.isna(y):
            vals.append(np.nan)
            means.append(np.nan)
            continue
        
        xi = to_idx(x, x_min, x_max, nx)
        yi = to_idx(y, y_min, y_max, ny)
        
        vals.append(float(sand_map[yi, xi]))
        
        # 3x3 neighborhood
        y0, y1 = max(0, yi-1), min(ny, yi+2)
        x0, x1 = max(0, xi-1), min(nx, xi+2)
        means.append(float(sand_map[y0:y1, x0:x1].mean()))
    
    df["sand_prop"] = vals
    df["sand_prop_local_mean"] = means
    return df

# Get coordinate ranges
all_x = pd.concat([X_prod["X"], X_pre["X"]])
all_y = pd.concat([X_prod["Y"], X_pre["Y"]])
x_min, x_max = all_x.min(), all_x.max()
y_min, y_max = all_y.min(), all_y.max()

X_prod = add_sand_features(X_prod, sand_map, x_min, x_max, y_min, y_max)
X_pre = add_sand_features(X_pre, sand_map, x_min, x_max, y_min, y_max)

print("Sand features added.")

### 4.4 Multi-Scale Spatial kNN Features

In [ ]:
def add_spatial_knn_features(X_prod, X_pre, k_list=[5, 10, 20, 50]):
    """Add multi-scale spatial kNN features."""
    all_df = pd.concat([X_prod.copy(), X_pre.copy()], ignore_index=True)
    
    coords = all_df[["X", "Y"]].astype(float).values.copy()
    col_means = np.nanmean(coords, axis=0)
    inds_nan = np.isnan(coords)
    coords[inds_nan] = np.take(col_means, np.where(inds_nan)[1])
    
    # Key features for neighbor aggregation
    base_cols = [c for c in ["sand_prop", "sand_prop_local_mean", "Vp_mean", "Vs_mean", "poisson_mean"] 
                 if c in all_df.columns]
    
    max_k = max(k_list)
    nn = NearestNeighbors(n_neighbors=min(max_k + 1, len(all_df)), metric="euclidean")
    nn.fit(coords)
    dists, idxs = nn.kneighbors(coords)
    
    all_neigh_idxs = idxs[:, 1:]  # Exclude self
    all_neigh_dists = dists[:, 1:]
    
    for k in k_list:
        neigh_idxs = all_neigh_idxs[:, :k]
        neigh_dists = all_neigh_dists[:, :k]
        
        all_df[f"knn{k}_dist_mean"] = neigh_dists.mean(axis=1)
        all_df[f"knn{k}_dist_std"] = neigh_dists.std(axis=1)
        
        for c in base_cols:
            vals = all_df[c].astype(float).values
            neigh_vals = vals[neigh_idxs]
            all_df[f"knn{k}_{c}_mean"] = np.nanmean(neigh_vals, axis=1)
            all_df[f"knn{k}_{c}_std"] = np.nanstd(neigh_vals, axis=1)
    
    X_prod_out = all_df.iloc[:len(X_prod)].reset_index(drop=True)
    X_pre_out = all_df.iloc[len(X_prod):].reset_index(drop=True)
    
    return X_prod_out, X_pre_out

X_prod, X_pre = add_spatial_knn_features(X_prod, X_pre)
print(f"Final production features: {X_prod.shape}")
print(f"Final pre-production features: {X_pre.shape}")

## 5. Prepare Training Data

In [ ]:
# Merge with targets
train_df = X_prod.merge(y_df, on="Well_ID", how="inner")

# Separate features and target
feature_cols = [c for c in train_df.columns if c not in ["Well_ID", "y_3yr_cum_oil"]]
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df["y_3yr_cum_oil"].values.astype(np.float32)
groups = train_df["Well_ID"].values

# Handle NaN
X_train = np.nan_to_num(X_train, nan=0.0)

print(f"Training data: X={X_train.shape}, y={y_train.shape}")

## 6. Model Training - XGBoost + LightGBM + ExtraTrees Ensemble

In [ ]:
# Cross-validation setup
gkf = GroupKFold(n_splits=5)

# Storage for OOF predictions
oof_xgb = np.zeros(len(y_train))
oof_lgb = np.zeros(len(y_train))
oof_et = np.zeros(len(y_train))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = np.log1p(y_train[tr_idx]), np.log1p(y_train[va_idx])
    
    # XGBoost
    xgb_model = xgb.XGBRegressor(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx] = np.expm1(xgb_model.predict(X_va))
    
    # LightGBM
    lgb_model = lgb.LGBMRegressor(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    oof_lgb[va_idx] = np.expm1(lgb_model.predict(X_va))
    
    # ExtraTrees
    et_model = ExtraTreesRegressor(n_estimators=300, max_depth=10, random_state=RANDOM_STATE)
    et_model.fit(X_tr, y_tr)
    oof_et[va_idx] = np.expm1(et_model.predict(X_va))
    
    print(f"Fold {fold+1} complete")

print("\nTraining complete!")

## 7. Stacking with RidgeCV

In [ ]:
# Stack OOF predictions
stack_X = np.column_stack([oof_xgb, oof_lgb, oof_et])

# Meta-learner
meta = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0])
meta.fit(stack_X, y_train)

# Final stacked predictions
oof_stacked = meta.predict(stack_X)

print(f"Meta-learner weights: {meta.coef_}")
print(f"Selected alpha: {meta.alpha_}")

## 8. Evaluation

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("=" * 50)
print("MODEL PERFORMANCE (OOF RMSE)")
print("=" * 50)
print(f"XGBoost:      {rmse(y_train, oof_xgb):,.0f} BBL")
print(f"LightGBM:     {rmse(y_train, oof_lgb):,.0f} BBL")
print(f"ExtraTrees:   {rmse(y_train, oof_et):,.0f} BBL")
print(f"\nStacked Ensemble: {rmse(y_train, oof_stacked):,.0f} BBL")
print(f"R2 Score: {r2_score(y_train, oof_stacked):.3f}")
print("=" * 50)

## 9. Visualize Predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
ax = axes[0]
ax.scatter(y_train, oof_stacked, alpha=0.5, edgecolor='k', linewidth=0.5)
ax.plot([0, y_train.max()], [0, y_train.max()], 'r--', label='Perfect')
ax.set_xlabel('Actual (BBL)')
ax.set_ylabel('Predicted (BBL)')
ax.set_title('Actual vs Predicted')
ax.legend()

# Residual distribution
ax = axes[1]
residuals = y_train - oof_stacked
ax.hist(residuals, bins=30, edgecolor='black')
ax.axvline(0, color='r', linestyle='--')
ax.set_xlabel('Residual (BBL)')
ax.set_ylabel('Frequency')
ax.set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## 10. Train Final Models & Generate Predictions

In [ ]:
# Train on full data
y_log = np.log1p(y_train)

final_xgb = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE)
final_lgb = lgb.LGBMRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, random_state=RANDOM_STATE, verbose=-1)
final_et = ExtraTreesRegressor(n_estimators=300, max_depth=10, random_state=RANDOM_STATE)

final_xgb.fit(X_train, y_log)
final_lgb.fit(X_train, y_log)
final_et.fit(X_train, y_log)

print("Final models trained!")

In [ ]:
# Prepare test data
test_feature_cols = [c for c in X_pre.columns if c != "Well_ID"]
X_test = X_pre[test_feature_cols].values.astype(np.float32)
X_test = np.nan_to_num(X_test, nan=0.0)

# Generate predictions
pred_xgb = np.expm1(final_xgb.predict(X_test))
pred_lgb = np.expm1(final_lgb.predict(X_test))
pred_et = np.expm1(final_et.predict(X_test))

# Stack
test_stack = np.column_stack([pred_xgb, pred_lgb, pred_et])
final_predictions = meta.predict(test_stack)
final_predictions = np.clip(final_predictions, 0, None)  # No negative production

print(f"Predictions generated for {len(final_predictions)} wells")
print(f"Prediction range: {final_predictions.min():,.0f} - {final_predictions.max():,.0f} BBL")

## 11. Save Results

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({
    "Well_ID": X_pre["Well_ID"],
    "Predicted_Oil_BBL": final_predictions
})

# Save
submission.to_csv("solutions.csv", index=False)
print("Predictions saved to solutions.csv")
submission.head()

---
## Summary

This notebook demonstrates our complete pipeline:
1. **Physics-based feature engineering** (Vp/Vs, Poisson, elastic moduli)
2. **Spatial feature learning** (Multi-scale kNN aggregation)
3. **Ensemble modeling** (XGBoost + LightGBM + ExtraTrees)
4. **Stacking** (RidgeCV meta-learner)

**Expected performance: ~1.95M BBL RMSE**